## 각 아이템별로 자신이 속한 메타데이터 조합의 주차별 median 값을 할당하여 데이터프레임으로 변환, 저장

### brand & main_color & category

In [28]:
import pandas as pd
import numpy as np
import pickle

# 데이터 로드
with open("/home/jihwan/fashion/data/train_val_test_split/train_list_240916.pkl", 'rb') as f:
    train = pickle.load(f)

with open("/home/jihwan/fashion/data/train_val_test_split/test_list_240916.pkl" , 'rb') as f:
    test = list(pickle.load(f))

with open("/home/jihwan/fashion/data/train_val_test_split/val_list_240916.pkl" , 'rb') as f:
    valid = list(pickle.load(f))    

meta = pd.read_csv("/home/jihwan/fashion/data/total_meta_data_str_value_240916.csv")
meta['item_number_color'] = meta['Unnamed: 0']
meta.drop(['color_full_spell', 'Unnamed: 0'], axis=1, inplace=True)

concat_sales_df = pd.read_csv("/home/jihwan/fashion/data/total_r_s.csv")
product_codes = pd.read_csv("/home/jihwan/fashion/data/product_codes.csv", encoding='cp949')
weeks = ['week'+str(i+1) for i in range(0, 12)]

# 데이터 전처리
concat_sales_df_a = concat_sales_df[['item_number_color', 'release_date']]
concat_sales_df_b = concat_sales_df.drop(['item_number_color', 'release_date'], axis=1)
concat_sales_df_b.columns = weeks
concat_sales_df = pd.concat([concat_sales_df_a, concat_sales_df_b], axis=1)
concat_sales_df['brand'] = concat_sales_df['item_number_color'].apply(lambda x: str(x)[0])

preprocessed_sales = concat_sales_df.drop(['release_date'], axis=1)
preprocessed_sales = preprocessed_sales.merge(meta, on='item_number_color')

preprocessed_sales = preprocessed_sales.merge(product_codes, on='category')
preprocessed_sales['category'] = preprocessed_sales['meaning']
preprocessed_sales = preprocessed_sales.drop(['meaning'], axis=1)

preprocessed_sales['combination'] = preprocessed_sales['brand'] +"_"+ preprocessed_sales['main_color'] +"_"+ preprocessed_sales['category'] #조합 구성
preprocessed_sales.drop(['brand',	'main_color',	'fabric',	'category'], axis=1, inplace=True) 

#train, valid, test 분리
train_name_df = pd.DataFrame({"item_number_color": train})
train_df = preprocessed_sales.merge(train_name_df, on="item_number_color")

valid_name_df = pd.DataFrame({"item_number_color":valid})
valid_df = preprocessed_sales.merge(valid_name_df, on="item_number_color")

test_name_df = pd.DataFrame({"item_number_color":test})
test_df = preprocessed_sales.merge(test_name_df, on="item_number_color")


# train에서 'brand', 'category' 조합에 대한 주차별 median 값 계산
grouped = train_df.groupby(['combination'])[weeks].median().reset_index()

## test에 train, valid 합친 데이터의 미디언을 넣기 위하여 새로 만든 그룹
grouped2 = pd.concat([train_df, valid_df],axis=0).groupby(['combination'])[weeks].median().reset_index()

# valid에서 'brand', 'main_color', 'category' 조합에 대한 주차별 median 값 계산
grouped3 = valid_df.groupby(['combination'])[weeks].median().reset_index()

# train에서 각 item_number_color에 대해 해당 조합의 median 값과 조합 정보 할당
train_df = train_df[['item_number_color', 'combination']].merge(grouped, on=['combination'], how='left', suffixes=('', '_median'))

# train 결과 확인 및 저장
print(train_df.shape)
train_df.to_csv("/home/jihwan/fashion/data/median_of_each_combination/brand&maincolor&category/train_data/train_median_of_each_combination_(brand&maincolor&category).csv", index=False)


###### valid, test에는 각 아이템이 속한 메타데이터 조합에 대해 train에서의 메타데이터 조합과 일치하는 경우 train에서의 그 조합의 주차별 값을 할당함. #######

# valid 데이터에 대해 train의 메타데이터 조합과 일치하는 경우 train의 주차별 median값을 할당
valid_df = valid_df[['item_number_color', 'combination']].merge(grouped3, on=['combination'], how='left', suffixes=('', '_median'))


### 결측치를 제외한 나머지 행들의 각 주차별 평균을 결측치에 넣음
# missing_value의 인덱스를 제외한 나머지 행들의 각 주차별 평균 계산
week_means = valid_df[~valid_df.index.isin([117, 125, 706, 960])][weeks].mean().round(1)


# 결측치가 있는 행들의 week1~week12 값을 평균값으로 대체
for index in [117, 125, 706, 960]:
    valid_df.loc[index, weeks] = week_means.values

# valid 결과 확인 및 저장
print(valid_df.shape)
valid_df.to_csv("/home/jihwan/fashion/data/median_of_each_combination/brand&maincolor&category/valid_data_towhich_train_value_was_added/valid_median_of_each_combination_(brand&maincolor&category).csv", index=False)

# test 데이터에 대해 train의 메타데이터 조합과 일치하는 경우 주차별 값을 할당
test_df = test_df[['item_number_color', 'combination']].merge(grouped2, on=['combination'], how='left',  suffixes=('', '_median'))

### 결측치를 제외한 나머지 행들의 각 주차별 평균을 결측치에 넣음
# missing_value의 인덱스를 제외한 나머지 행들의 각 주차별 평균 계산
week_means = test_df[~test_df.index.isin([638, 643, 826, 832, 2094, 2099, 2102])][weeks].mean().round(1)

# 결측치가 있는 행들의 week1~week12 값을 평균값으로 대체
for index in [638, 643, 826, 832, 2094, 2099, 2102]:
    test_df.loc[index, weeks] = week_means.values

# test 결과 확인 및 저장
print(test_df.shape)
test_df.to_csv("/home/jihwan/fashion/data/median_of_each_combination/brand&maincolor&category/test_data_towhich_train_value_was_added/test_median_of_each_combination_(brand&maincolor&category).csv", index=False)


(14757, 14)
(1919, 14)
(2822, 14)


In [29]:
set(valid_df.combination).difference(set(train_df.combination)) #valid에는 4개의 조합이 train에 없음 - 결측치를 각 주차의 median들의 평균값으로 대체함

{'J_BEIGE_DOWN VEST', 'J_BROWN_VEST', 'J_GOLD_JACKET', 'M_MINT_DOWN VEST'}

In [30]:
set(test_df.combination).difference(set(train_df.combination)) #test에는 5개의 조합이 train에 없음 - 결측치를 각 주차의 median들의 평균값으로 대체함

{'J_BEIGE_DOWN VEST',
 'J_PINK_VEST',
 'J_WHITE_VEST',
 'M_BROWN_DENIM',
 'M_PURPLE_DENIM'}

In [31]:
#Train 처리 결과
train_after = pd.read_csv("/home/jihwan/fashion/data/median_of_each_combination/brand&maincolor&category/train_data/train_median_of_each_combination_(brand&maincolor&category).csv")
print(train_after.isnull().sum())
train_after

item_number_color    0
combination          0
week1                0
week2                0
week3                0
week4                0
week5                0
week6                0
week7                0
week8                0
week9                0
week10               0
week11               0
week12               0
dtype: int64


,item_number_color,combination,week1,week2,week3,week4,week5,week6,week7,week8,week9,week10,week11,week12
0,JOBL16BAPK,J_PINK_BLOUSE,5.0,7.0,6.0,9.0,5.0,9.0,7.0,8.0,8.0,8.0,5.0,4.0
1,JOBL26FADN,J_BLUE_BLOUSE,6.0,8.0,7.0,8.0,7.0,7.0,6.0,5.0,6.0,5.0,5.0,5.0
2,JOBL26FAWH,J_WHITE_BLOUSE,4.0,7.0,5.0,6.0,4.0,6.0,5.0,6.0,6.5,7.0,6.0,6.0
3,JOBL361AOR,J_ORANGE_BLOUSE,2.0,5.0,1.5,1.5,1.5,2.0,5.5,3.5,4.5,3.0,0.5,2.0
4,JOBL361AWH,J_WHITE_BLOUSE,4.0,7.0,5.0,6.0,4.0,6.0,5.0,6.0,6.5,7.0,6.0,6.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
14752,MWWS0101BK,M_BLACK_W/SHIRTS,7.0,13.0,19.0,20.0,29.0,41.0,38.0,40.0,37.0,48.0,46.0,37.0
14753,MWWS0101DL,M_BLUE_W/SHIRTS,17.0,32.0,39.0,48.0,46.0,54.0,60.0,60.0,59.0,55.0,52.0,47.0
14754,MWWS0101LG,M_GREY_W/SHIRTS,21.0,42.0,46.0,62.0,67.0,81.0,82.0,69.0,53.0,58.0,60.0,52.0
14755,MWWS0102TB,M_BLUE_W/SHIRTS,17.0,32.0,39.0,48.0,46.0,54.0,60.0,60.0,59.0,55.0,52.0,47.0


In [32]:
#Valid 처리 결과 - Train의 값이 들어가야 함
valid_after = pd.read_csv("/home/jihwan/fashion/data/median_of_each_combination/brand&maincolor&category/valid_data_towhich_train_value_was_added/valid_median_of_each_combination_(brand&maincolor&category).csv")
print(valid_after.isnull().sum())
valid_after

item_number_color    0
combination          0
week1                0
week2                0
week3                0
week4                0
week5                0
week6                0
week7                0
week8                0
week9                0
week10               0
week11               0
week12               0
dtype: int64


,item_number_color,combination,week1,week2,week3,week4,week5,week6,week7,week8,week9,week10,week11,week12
0,JVRF726AKH,J_KHAKI_RF,8.0,12.5,12.0,6.5,4.5,4.0,2.0,3.5,1.0,1.0,0.0,1.5
1,JWBL221ABL,J_BLUE_BLOUSE,14.0,28.0,20.0,14.0,14.0,12.0,14.0,9.0,10.0,9.0,7.0,8.0
2,JWBL226ANA,J_BLUE_BLOUSE,14.0,28.0,20.0,14.0,14.0,12.0,14.0,9.0,10.0,9.0,7.0,8.0
3,JWBL322ANA,J_BLUE_BLOUSE,14.0,28.0,20.0,14.0,14.0,12.0,14.0,9.0,10.0,9.0,7.0,8.0
4,JWBL322CNA,J_BLUE_BLOUSE,14.0,28.0,20.0,14.0,14.0,12.0,14.0,9.0,10.0,9.0,7.0,8.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1914,MXWS0150MT,M_MINT_W/SHIRTS,15.5,45.0,52.0,91.0,110.0,112.0,95.0,79.5,71.5,97.5,72.0,61.5
1915,MXWS0150SB,M_BLUE_W/SHIRTS,8.0,21.0,31.0,37.0,49.0,56.0,65.0,50.0,49.0,48.0,29.0,35.0
1916,MXWS0150WH,M_WHITE_W/SHIRTS,7.0,20.0,25.0,29.0,34.0,49.0,45.0,48.0,44.0,41.0,39.0,27.0
1917,MXWS0150WY,M_YELLOW_W/SHIRTS,1.0,0.0,1.0,0.0,0.0,0.0,0.0,28.0,31.0,37.0,43.0,31.0


In [33]:
# valid에서 결측치가 있는 행 확인 - valid에는 존재하는 4개의 조합이 train에 없는데, 이 조합으로 구성된 valid의 행이 총 4개
missing_rows_val = valid_after[valid_after.isnull().any(axis=1)]
missing_rows_val

,item_number_color,combination,week1,week2,week3,week4,week5,week6,week7,week8,week9,week10,week11,week12


In [34]:
valid_after.isnull().sum()

item_number_color    0
combination          0
week1                0
week2                0
week3                0
week4                0
week5                0
week6                0
week7                0
week8                0
week9                0
week10               0
week11               0
week12               0
dtype: int64

In [35]:
#Test 처리 결과 - Train의 값이 들어가야 함
test_after = pd.read_csv("/home/jihwan/fashion/data/median_of_each_combination/brand&maincolor&category/test_data_towhich_train_value_was_added/test_median_of_each_combination_(brand&maincolor&category).csv")
print(test_after.isnull().sum())
test_after

item_number_color    0
combination          0
week1                0
week2                0
week3                0
week4                0
week5                0
week6                0
week7                0
week8                0
week9                0
week10               0
week11               0
week12               0
dtype: int64


,item_number_color,combination,week1,week2,week3,week4,week5,week6,week7,week8,week9,week10,week11,week12
0,JXBL121ALY,J_YELLOW_BLOUSE,4.0,9.5,8.0,7.0,6.5,7.5,6.5,5.0,6.0,5.0,4.5,5.0
1,JXBL121BLC,J_PURPLE_BLOUSE,9.0,14.0,15.0,10.0,14.0,11.0,11.0,6.0,7.0,7.0,6.0,7.0
2,JXBL121BOW,J_WHITE_BLOUSE,4.0,7.0,5.0,6.0,4.0,6.0,5.0,6.0,7.0,7.0,6.0,6.0
3,JXBL121CYE,J_YELLOW_BLOUSE,4.0,9.5,8.0,7.0,6.5,7.5,6.5,5.0,6.0,5.0,4.5,5.0
4,JXBL321AIN,J_BLUE_BLOUSE,6.0,8.0,8.0,8.5,7.5,8.0,7.0,6.0,6.0,5.5,5.0,5.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2817,MYWS3221OR,M_ORANGE_W/SHIRTS,20.0,34.0,46.0,48.0,42.0,49.0,44.0,37.0,41.0,41.0,33.0,33.0
2818,MYWS3230BR,M_BROWN_W/SHIRTS,21.5,38.0,38.0,46.5,67.5,64.0,63.5,54.5,48.0,59.5,42.0,33.0
2819,MYWS3230CH,M_BLACK_W/SHIRTS,11.0,18.0,29.0,39.0,42.0,45.0,52.0,52.0,44.0,51.0,45.0,36.0
2820,MYWS3231BL,M_BLUE_W/SHIRTS,15.5,31.5,37.0,46.5,46.5,54.0,60.5,58.0,58.5,54.0,50.0,46.0


In [36]:
# test에서 결측치가 있는 행 확인 - test에는 존재하는 5개의 조합이 train에 없는데, 이 조합으로 구성된 test의 행이 총 8개
missing_rows_test = test_after[test_after.isnull().any(axis=1)]
missing_rows_test

,item_number_color,combination,week1,week2,week3,week4,week5,week6,week7,week8,week9,week10,week11,week12


In [37]:
test_after.isnull().sum()

item_number_color    0
combination          0
week1                0
week2                0
week3                0
week4                0
week5                0
week6                0
week7                0
week8                0
week9                0
week10               0
week11               0
week12               0
dtype: int64

In [38]:
# null 값이 포함된 행의 인덱스 찾기
null_indices = test_after[test_after.isnull().any(axis=1)].index
print("null indices:",null_indices)

null indices: Index([], dtype='int64')


### brand & category

In [65]:
import pandas as pd
import numpy as np
import pickle

# 데이터 로드
with open("/home/jihwan/fashion/data/train_val_test_split/train_list_240916.pkl", 'rb') as f:
    train = pickle.load(f)

with open("/home/jihwan/fashion/data/train_val_test_split/test_list_240916.pkl" , 'rb') as f:
    test = list(pickle.load(f))

with open("/home/jihwan/fashion/data/train_val_test_split/val_list_240916.pkl" , 'rb') as f:
    valid = list(pickle.load(f))    

meta = pd.read_csv("/home/jihwan/fashion/data/total_meta_data_str_value_240916.csv")
meta['item_number_color'] = meta['Unnamed: 0']
meta.drop(['color_full_spell', 'Unnamed: 0'], axis=1, inplace=True)

concat_sales_df = pd.read_csv("/home/jihwan/fashion/data/total_r_s.csv")
product_codes = pd.read_csv("/home/jihwan/fashion/data/product_codes.csv", encoding='cp949')
weeks = ['week'+str(i+1) for i in range(0, 12)]

# 데이터 전처리
concat_sales_df_a = concat_sales_df[['item_number_color', 'release_date']]
concat_sales_df_b = concat_sales_df.drop(['item_number_color', 'release_date'], axis=1)
concat_sales_df_b.columns = weeks
concat_sales_df = pd.concat([concat_sales_df_a, concat_sales_df_b], axis=1)
concat_sales_df['brand'] = concat_sales_df['item_number_color'].apply(lambda x: str(x)[0])

preprocessed_sales = concat_sales_df.drop(['release_date'], axis=1)
preprocessed_sales = preprocessed_sales.merge(meta, on='item_number_color')

preprocessed_sales = preprocessed_sales.merge(product_codes, on='category')
preprocessed_sales['category'] = preprocessed_sales['meaning']
preprocessed_sales = preprocessed_sales.drop(['meaning'], axis=1)

preprocessed_sales['combination'] = preprocessed_sales['brand'] +"_"+ preprocessed_sales['category'] #조합 구성
preprocessed_sales.drop(['brand',	'main_color',	'fabric',	'category'], axis=1, inplace=True) 

#train, valid, test 분리
train_name_df = pd.DataFrame({"item_number_color": train})
train_df = preprocessed_sales.merge(train_name_df, on="item_number_color")

valid_name_df = pd.DataFrame({"item_number_color":valid})
valid_df = preprocessed_sales.merge(valid_name_df, on="item_number_color")

test_name_df = pd.DataFrame({"item_number_color":test})
test_df = preprocessed_sales.merge(test_name_df, on="item_number_color")


# train에서 'brand', 'category' 조합에 대한 주차별 median 값 계산
grouped = train_df.groupby(['combination'])[weeks].median().reset_index()

## test에 train, valid 합친 데이터의 미디언을 넣기 위하여 새로 만든 그룹
grouped2 = pd.concat([train_df, valid_df],axis=0).groupby(['combination'])[weeks].median().reset_index()

# valid에서 'brand', 'fabric' 조합에 대한 주차별 median 값 계산
grouped3 = valid_df.groupby(['combination'])[weeks].median().reset_index()

# train에서 각 item_number_color에 대해 해당 조합의 median 값과 조합 정보 할당
train_df = train_df[['item_number_color', 'combination']].merge(grouped, on=['combination'], how='left',suffixes=('', '_median'))

# train 결과 확인 및 저장
print(train_df.shape)
train_df.to_csv("/home/jihwan/fashion/data/median_of_each_combination/brand&category/train_data/train_median_of_each_combination_(brand&category).csv", index=False)


###### valid, test에는 각 아이템이 속한 메타데이터 조합에 대해 train에서의 메타데이터 조합과 일치하는 경우 train에서의 그 조합의 주차별 값을 할당함. #######

# valid 데이터에 대해 train의 메타데이터 조합과 일치하는 경우 train의 주차별 median값을 할당
valid_df = valid_df[['item_number_color', 'combination']].merge(grouped3, on=['combination'], how='left',suffixes=('', '_median'))

# valid 결과 확인 및 저장
print(valid_df.shape)
valid_df.to_csv("/home/jihwan/fashion/data/median_of_each_combination/brand&category/valid_data_towhich_train_value_was_added/valid_median_of_each_combination_(brand&category).csv", index=False)

# test 데이터에 대해 train의 메타데이터 조합과 일치하는 경우 주차별 값을 할당
test_df = test_df[['item_number_color', 'combination']].merge(grouped, on=['combination'], how='left',  suffixes=('', '_median'))

# test 결과 확인 및 저장
print(test_df.shape)
test_df.to_csv("/home/jihwan/fashion/data/median_of_each_combination/brand&category/test_data_towhich_train_value_was_added/test_median_of_each_combination_(brand&category).csv", index=False)


(14757, 14)
(1919, 14)
(2822, 14)


In [66]:
#Train 처리 결과
train_after = pd.read_csv("/home/jihwan/fashion/data/median_of_each_combination/brand&category/train_data/train_median_of_each_combination_(brand&category).csv")
print(train_after.isnull().sum())
train_after

item_number_color    0
combination          0
week1                0
week2                0
week3                0
week4                0
week5                0
week6                0
week7                0
week8                0
week9                0
week10               0
week11               0
week12               0
dtype: int64


,item_number_color,combination,week1,week2,week3,week4,week5,week6,week7,week8,week9,week10,week11,week12
0,JOBL16BAPK,J_BLOUSE,6.0,9.0,8.0,8.0,7.0,7.0,7.0,6.0,7.0,6.0,5.0,5.0
1,JOBL26FADN,J_BLOUSE,6.0,9.0,8.0,8.0,7.0,7.0,7.0,6.0,7.0,6.0,5.0,5.0
2,JOBL26FAWH,J_BLOUSE,6.0,9.0,8.0,8.0,7.0,7.0,7.0,6.0,7.0,6.0,5.0,5.0
3,JOBL361AOR,J_BLOUSE,6.0,9.0,8.0,8.0,7.0,7.0,7.0,6.0,7.0,6.0,5.0,5.0
4,JOBL361AWH,J_BLOUSE,6.0,9.0,8.0,8.0,7.0,7.0,7.0,6.0,7.0,6.0,5.0,5.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
14752,MWWS0101BK,M_W/SHIRTS,14.0,26.0,34.0,41.0,43.0,51.0,55.0,55.0,51.0,49.0,46.0,43.0
14753,MWWS0101DL,M_W/SHIRTS,14.0,26.0,34.0,41.0,43.0,51.0,55.0,55.0,51.0,49.0,46.0,43.0
14754,MWWS0101LG,M_W/SHIRTS,14.0,26.0,34.0,41.0,43.0,51.0,55.0,55.0,51.0,49.0,46.0,43.0
14755,MWWS0102TB,M_W/SHIRTS,14.0,26.0,34.0,41.0,43.0,51.0,55.0,55.0,51.0,49.0,46.0,43.0


In [67]:
#Valid 처리 결과 - Train의 값이 들어가야 함
valid_after = pd.read_csv("/home/jihwan/fashion/data/median_of_each_combination/brand&category/valid_data_towhich_train_value_was_added/valid_median_of_each_combination_(brand&category).csv")
print(valid_after.isnull().sum())
valid_after

item_number_color    0
combination          0
week1                0
week2                0
week3                0
week4                0
week5                0
week6                0
week7                0
week8                0
week9                0
week10               0
week11               0
week12               0
dtype: int64


,item_number_color,combination,week1,week2,week3,week4,week5,week6,week7,week8,week9,week10,week11,week12
0,JVRF726AKH,J_RF,4.0,9.0,8.0,9.0,6.0,6.0,7.0,6.0,4.0,4.0,3.0,4.0
1,JWBL221ABL,J_BLOUSE,9.0,7.0,15.0,13.0,12.0,12.0,10.0,9.0,10.0,4.0,7.0,4.0
2,JWBL226ANA,J_BLOUSE,9.0,7.0,15.0,13.0,12.0,12.0,10.0,9.0,10.0,4.0,7.0,4.0
3,JWBL322ANA,J_BLOUSE,9.0,7.0,15.0,13.0,12.0,12.0,10.0,9.0,10.0,4.0,7.0,4.0
4,JWBL322CNA,J_BLOUSE,9.0,7.0,15.0,13.0,12.0,12.0,10.0,9.0,10.0,4.0,7.0,4.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1914,MXWS0150MT,M_W/SHIRTS,13.0,28.0,37.0,45.0,57.0,57.0,65.0,55.0,54.0,46.0,38.0,35.0
1915,MXWS0150SB,M_W/SHIRTS,13.0,28.0,37.0,45.0,57.0,57.0,65.0,55.0,54.0,46.0,38.0,35.0
1916,MXWS0150WH,M_W/SHIRTS,13.0,28.0,37.0,45.0,57.0,57.0,65.0,55.0,54.0,46.0,38.0,35.0
1917,MXWS0150WY,M_W/SHIRTS,13.0,28.0,37.0,45.0,57.0,57.0,65.0,55.0,54.0,46.0,38.0,35.0


In [68]:
#Test 처리 결과 - Train의 값이 들어가야 함
test_after = pd.read_csv("/home/jihwan/fashion/data/median_of_each_combination/brand&category/test_data_towhich_train_value_was_added/test_median_of_each_combination_(brand&category).csv")
print(test_after.isnull().sum())
test_after

item_number_color    0
combination          0
week1                0
week2                0
week3                0
week4                0
week5                0
week6                0
week7                0
week8                0
week9                0
week10               0
week11               0
week12               0
dtype: int64


,item_number_color,combination,week1,week2,week3,week4,week5,week6,week7,week8,week9,week10,week11,week12
0,JXBL121ALY,J_BLOUSE,6.0,9.0,8.0,8.0,7.0,7.0,7.0,6.0,7.0,6.0,5.0,5.0
1,JXBL121BLC,J_BLOUSE,6.0,9.0,8.0,8.0,7.0,7.0,7.0,6.0,7.0,6.0,5.0,5.0
2,JXBL121BOW,J_BLOUSE,6.0,9.0,8.0,8.0,7.0,7.0,7.0,6.0,7.0,6.0,5.0,5.0
3,JXBL121CYE,J_BLOUSE,6.0,9.0,8.0,8.0,7.0,7.0,7.0,6.0,7.0,6.0,5.0,5.0
4,JXBL321AIN,J_BLOUSE,6.0,9.0,8.0,8.0,7.0,7.0,7.0,6.0,7.0,6.0,5.0,5.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2817,MYWS3221OR,M_W/SHIRTS,14.0,26.0,34.0,41.0,43.0,51.0,55.0,55.0,51.0,49.0,46.0,43.0
2818,MYWS3230BR,M_W/SHIRTS,14.0,26.0,34.0,41.0,43.0,51.0,55.0,55.0,51.0,49.0,46.0,43.0
2819,MYWS3230CH,M_W/SHIRTS,14.0,26.0,34.0,41.0,43.0,51.0,55.0,55.0,51.0,49.0,46.0,43.0
2820,MYWS3231BL,M_W/SHIRTS,14.0,26.0,34.0,41.0,43.0,51.0,55.0,55.0,51.0,49.0,46.0,43.0


# brand & fabric & category

In [60]:
import pandas as pd
import numpy as np
import pickle

# 데이터 로드
with open("/home/jihwan/fashion/data/train_val_test_split/train_list_240916.pkl", 'rb') as f:
    train = pickle.load(f)

with open("/home/jihwan/fashion/data/train_val_test_split/test_list_240916.pkl" , 'rb') as f:
    test = list(pickle.load(f))

with open("/home/jihwan/fashion/data/train_val_test_split/val_list_240916.pkl" , 'rb') as f:
    valid = list(pickle.load(f))    

meta = pd.read_csv("/home/jihwan/fashion/data/total_meta_data_str_value_240916.csv")
meta['item_number_color'] = meta['Unnamed: 0']
meta.drop(['color_full_spell', 'Unnamed: 0'], axis=1, inplace=True)

concat_sales_df = pd.read_csv("/home/jihwan/fashion/data/total_r_s.csv")
product_codes = pd.read_csv("/home/jihwan/fashion/data/product_codes.csv", encoding='cp949')
weeks = ['week'+str(i+1) for i in range(0, 12)]

# 데이터 전처리
concat_sales_df_a = concat_sales_df[['item_number_color', 'release_date']]
concat_sales_df_b = concat_sales_df.drop(['item_number_color', 'release_date'], axis=1)
concat_sales_df_b.columns = weeks
concat_sales_df = pd.concat([concat_sales_df_a, concat_sales_df_b], axis=1)
concat_sales_df['brand'] = concat_sales_df['item_number_color'].apply(lambda x: str(x)[0])

preprocessed_sales = concat_sales_df.drop(['release_date'], axis=1)
preprocessed_sales = preprocessed_sales.merge(meta, on='item_number_color')

preprocessed_sales = preprocessed_sales.merge(product_codes, on='category')
preprocessed_sales['category'] = preprocessed_sales['meaning']
preprocessed_sales = preprocessed_sales.drop(['meaning'], axis=1)

preprocessed_sales['combination'] = preprocessed_sales['brand'] +"_"+ preprocessed_sales['fabric'] +"_"+ preprocessed_sales['category'] #조합 구성
preprocessed_sales.drop(['brand',	'main_color',	'fabric',	'category'], axis=1, inplace=True) 

#train, valid, test 분리
train_name_df = pd.DataFrame({"item_number_color": train})
train_df = preprocessed_sales.merge(train_name_df, on="item_number_color")

valid_name_df = pd.DataFrame({"item_number_color":valid})
valid_df = preprocessed_sales.merge(valid_name_df, on="item_number_color")

test_name_df = pd.DataFrame({"item_number_color":test})
test_df = preprocessed_sales.merge(test_name_df, on="item_number_color")


# train에서 'brand', 'fabric', 'category' 조합에 대한 주차별 median 값 계산
grouped = train_df.groupby(['combination'])[weeks].median().reset_index()

## test에 train, valid 합친 데이터의 미디언을 넣기 위하여 새로 만든 그룹
grouped2 = pd.concat([train_df, valid_df],axis=0).groupby(['combination'])[weeks].median().reset_index()

# valid에서 'brand', 'fabric' 조합에 대한 주차별 median 값 계산
grouped3 = valid_df.groupby(['combination'])[weeks].median().reset_index()

# train에서 각 item_number_color에 대해 해당 조합의 median 값과 조합 정보 할당
train_df = train_df[['item_number_color', 'combination']].merge(grouped, on=['combination'], how='left', suffixes=('', '_median'))

# train 결과 확인 및 저장
print(train_df.shape)
train_df.to_csv("/home/jihwan/fashion/data/median_of_each_combination/brand&fabric&category/train_data/train_median_of_each_combination_(brand&fabric&category).csv", index=False)


###### valid, test에는 각 아이템이 속한 메타데이터 조합에 대해 train에서의 메타데이터 조합과 일치하는 경우 train에서의 그 조합의 주차별 값을 할당함. #######

# valid 데이터에 대해 train의 메타데이터 조합과 일치하는 경우 train의 주차별 median값을 할당
valid_df = valid_df[['item_number_color', 'combination']].merge(grouped3, on=['combination'], how='left', suffixes=('', '_median'))


# ### 결측치를 제외한 나머지 행들의 각 주차별 평균을 결측치에 넣음 - 처음에 주석처리해서 테이블 만들고 결측값 확인 후 주석 제거해서 결측값 인덱스 넣어서 다시 코드 실행해야 함
# # missing_value의 인덱스를 제외한 나머지 행들의 각 주차별 평균 계산
# week_means = valid_df[~valid_df.index.isin([16, 709, 729, 974, 975, 1852, 1853])][weeks].mean().round(1)

# # 결측치가 있는 행들의 week1~week12 값을 평균값으로 대체
# for index in [16, 709, 729, 974, 975, 1852, 1853]:
#     valid_df.loc[index, weeks] = week_means.values

# valid 결과 확인 및 저장
print(valid_df.shape)
valid_df.to_csv("/home/jihwan/fashion/data/median_of_each_combination/brand&fabric&category/valid_data_towhich_train_value_was_added/valid_median_of_each_combination_(brand&fabric&category).csv", index=False)

# test 데이터에 대해 train의 메타데이터 조합과 일치하는 경우 주차별 값을 할당
# test_df = test_df[['item_number_color', 'combination']].merge(grouped, on=['combination'], how='left',  suffixes=('', '_median'))
test_df = test_df[['item_number_color', 'combination']].merge(grouped, on=['combination'], how='left',  suffixes=('', '_median'))

### 결측치를 제외한 나머지 행들의 각 주차별 평균을 결측치에 넣음- 처음에 주석처리해서 테이블 만들고 결측값 확인 후 주석 제거해서 결측값 인덱스 넣어서 다시 코드 실행해야 함
# missing_value의 인덱스를 제외한 나머지 행들의 각 주차별 평균 계산
week_means = test_df[~test_df.index.isin([39, 40, 156, 160, 161, 640, 874, 875, 1583, 2093, 2134, 2135, 2450])][weeks].mean().round(1)

# 결측치가 있는 행들의 week1~week12 값을 평균값으로 대체
for index in [39, 40, 156, 160, 161, 640, 874, 875, 1583, 2093, 2134, 2135, 2450]:
    test_df.loc[index, weeks] = week_means.values

# test 결과 확인 및 저장
print(test_df.shape)
test_df.to_csv("/home/jihwan/fashion/data/median_of_each_combination/brand&fabric&category/test_data_towhich_train_value_was_added/test_median_of_each_combination_(brand&fabric&category).csv", index=False)


(14757, 14)
(1919, 14)
(2822, 14)


In [61]:
#Train 처리 결과
train_after = pd.read_csv("/home/jihwan/fashion/data/median_of_each_combination/brand&fabric&category/train_data/train_median_of_each_combination_(brand&fabric&category).csv")
print(train_after.isnull().sum())
train_after

item_number_color    0
combination          0
week1                0
week2                0
week3                0
week4                0
week5                0
week6                0
week7                0
week8                0
week9                0
week10               0
week11               0
week12               0
dtype: int64


,item_number_color,combination,week1,week2,week3,week4,week5,week6,week7,week8,week9,week10,week11,week12
0,JOBL16BAPK,J_WOVEN_BLOUSE,6.0,9.0,8.0,8.0,7.0,7.0,7.0,6.0,7.0,6.0,6.0,5.0
1,JOBL26FADN,J_WOVEN_BLOUSE,6.0,9.0,8.0,8.0,7.0,7.0,7.0,6.0,7.0,6.0,6.0,5.0
2,JOBL26FAWH,J_WOVEN_BLOUSE,6.0,9.0,8.0,8.0,7.0,7.0,7.0,6.0,7.0,6.0,6.0,5.0
3,JOBL361AOR,J_WOVEN_BLOUSE,6.0,9.0,8.0,8.0,7.0,7.0,7.0,6.0,7.0,6.0,6.0,5.0
4,JOBL361AWH,J_WOVEN_BLOUSE,6.0,9.0,8.0,8.0,7.0,7.0,7.0,6.0,7.0,6.0,6.0,5.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
14752,MWWS0101BK,M_WOVEN_W/SHIRTS,14.0,26.0,34.0,40.0,43.0,51.0,55.0,55.0,51.0,49.0,46.0,42.0
14753,MWWS0101DL,M_WOVEN_W/SHIRTS,14.0,26.0,34.0,40.0,43.0,51.0,55.0,55.0,51.0,49.0,46.0,42.0
14754,MWWS0101LG,M_WOVEN_W/SHIRTS,14.0,26.0,34.0,40.0,43.0,51.0,55.0,55.0,51.0,49.0,46.0,42.0
14755,MWWS0102TB,M_WOVEN_W/SHIRTS,14.0,26.0,34.0,40.0,43.0,51.0,55.0,55.0,51.0,49.0,46.0,42.0


In [62]:
#Valid 처리 결과 - Train의 값이 들어가야 함
valid_after = pd.read_csv("/home/jihwan/fashion/data/median_of_each_combination/brand&fabric&category/valid_data_towhich_train_value_was_added/valid_median_of_each_combination_(brand&fabric&category).csv")
print(valid_after.isnull().sum())

# null 값이 포함된 행의 인덱스 찾기
null_indices = valid_after[valid_after.isnull().any(axis=1)].index
print("null indices:",null_indices)

valid_after

item_number_color    0
combination          0
week1                0
week2                0
week3                0
week4                0
week5                0
week6                0
week7                0
week8                0
week9                0
week10               0
week11               0
week12               0
dtype: int64
null indices: Index([], dtype='int64')


,item_number_color,combination,week1,week2,week3,week4,week5,week6,week7,week8,week9,week10,week11,week12
0,JVRF726AKH,J_특종_RF,4.0,9.0,8.0,9.0,6.0,6.0,7.0,6.0,4.0,4.0,3.0,4.0
1,JWBL221ABL,J_WOVEN_BLOUSE,9.5,8.0,16.0,13.5,13.0,13.5,12.0,9.0,10.5,4.5,7.0,4.5
2,JWBL226ANA,J_WOVEN_BLOUSE,9.5,8.0,16.0,13.5,13.0,13.5,12.0,9.0,10.5,4.5,7.0,4.5
3,JWBL322ANA,J_WOVEN_BLOUSE,9.5,8.0,16.0,13.5,13.0,13.5,12.0,9.0,10.5,4.5,7.0,4.5
4,JWBL322CNA,J_WOVEN_BLOUSE,9.5,8.0,16.0,13.5,13.0,13.5,12.0,9.0,10.5,4.5,7.0,4.5
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1914,MXWS0150MT,M_WOVEN_W/SHIRTS,13.0,29.0,39.0,45.0,57.0,57.0,65.0,53.0,54.0,46.0,36.0,35.0
1915,MXWS0150SB,M_WOVEN_W/SHIRTS,13.0,29.0,39.0,45.0,57.0,57.0,65.0,53.0,54.0,46.0,36.0,35.0
1916,MXWS0150WH,M_WOVEN_W/SHIRTS,13.0,29.0,39.0,45.0,57.0,57.0,65.0,53.0,54.0,46.0,36.0,35.0
1917,MXWS0150WY,M_WOVEN_W/SHIRTS,13.0,29.0,39.0,45.0,57.0,57.0,65.0,53.0,54.0,46.0,36.0,35.0


In [63]:
# null 값이 포함된 행의 인덱스 찾기
null_indices = valid_after[valid_after.isnull().any(axis=1)].index
print("null indices:",null_indices)

null indices: Index([], dtype='int64')


In [64]:
#Test 처리 결과 - Train의 값이 들어가야 함
test_after = pd.read_csv("/home/jihwan/fashion/data/median_of_each_combination/brand&fabric&category/test_data_towhich_train_value_was_added/test_median_of_each_combination_(brand&fabric&category).csv")
print(test_after.isnull().sum())


# null 값이 포함된 행의 인덱스 찾기
null_indices = test_after[test_after.isnull().any(axis=1)].index
print("null indices:",null_indices)

test_after

item_number_color    0
combination          0
week1                0
week2                0
week3                0
week4                0
week5                0
week6                0
week7                0
week8                0
week9                0
week10               0
week11               0
week12               0
dtype: int64
null indices: Index([], dtype='int64')


,item_number_color,combination,week1,week2,week3,week4,week5,week6,week7,week8,week9,week10,week11,week12
0,JXBL121ALY,J_WOVEN_BLOUSE,6.0,9.0,8.0,8.0,7.0,7.0,7.0,6.0,7.0,6.0,6.0,5.0
1,JXBL121BLC,J_WOVEN_BLOUSE,6.0,9.0,8.0,8.0,7.0,7.0,7.0,6.0,7.0,6.0,6.0,5.0
2,JXBL121BOW,J_WOVEN_BLOUSE,6.0,9.0,8.0,8.0,7.0,7.0,7.0,6.0,7.0,6.0,6.0,5.0
3,JXBL121CYE,J_WOVEN_BLOUSE,6.0,9.0,8.0,8.0,7.0,7.0,7.0,6.0,7.0,6.0,6.0,5.0
4,JXBL321AIN,J_DENIM_BLOUSE,4.0,6.0,3.0,7.0,8.0,14.0,4.0,6.0,3.0,4.0,5.0,3.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2817,MYWS3221OR,M_WOVEN_W/SHIRTS,14.0,26.0,34.0,40.0,43.0,51.0,55.0,55.0,51.0,49.0,46.0,42.0
2818,MYWS3230BR,M_WOVEN_W/SHIRTS,14.0,26.0,34.0,40.0,43.0,51.0,55.0,55.0,51.0,49.0,46.0,42.0
2819,MYWS3230CH,M_WOVEN_W/SHIRTS,14.0,26.0,34.0,40.0,43.0,51.0,55.0,55.0,51.0,49.0,46.0,42.0
2820,MYWS3231BL,M_DENIM_W/SHIRTS,15.0,29.5,31.5,45.5,46.5,52.0,44.0,53.5,54.5,56.0,74.5,45.0


In [10]:
# null 값이 포함된 행의 인덱스 찾기
null_indices = test_after[test_after.isnull().any(axis=1)].index
print("null indices:",null_indices)

null indices: Index([39, 40, 156, 160, 161, 640, 874, 875, 1583, 2093, 2134, 2135, 2450], dtype='int64')


# brand & fabric

In [75]:
import pandas as pd
import numpy as np
import pickle

# 데이터 로드
with open("/home/jihwan/fashion/data/train_val_test_split/train_list_240916.pkl", 'rb') as f:
    train = pickle.load(f)

with open("/home/jihwan/fashion/data/train_val_test_split/test_list_240916.pkl" , 'rb') as f:
    test = list(pickle.load(f))

with open("/home/jihwan/fashion/data/train_val_test_split/val_list_240916.pkl" , 'rb') as f:
    valid = list(pickle.load(f))    

meta = pd.read_csv("/home/jihwan/fashion/data/total_meta_data_str_value_240916.csv")
meta['item_number_color'] = meta['Unnamed: 0']
meta.drop(['color_full_spell', 'Unnamed: 0'], axis=1, inplace=True)

concat_sales_df = pd.read_csv("/home/jihwan/fashion/data/total_r_s.csv")
product_codes = pd.read_csv("/home/jihwan/fashion/data/product_codes.csv", encoding='cp949')
weeks = ['week'+str(i+1) for i in range(0, 12)]

# 데이터 전처리
concat_sales_df_a = concat_sales_df[['item_number_color', 'release_date']]
concat_sales_df_b = concat_sales_df.drop(['item_number_color', 'release_date'], axis=1)
concat_sales_df_b.columns = weeks
concat_sales_df = pd.concat([concat_sales_df_a, concat_sales_df_b], axis=1)
concat_sales_df['brand'] = concat_sales_df['item_number_color'].apply(lambda x: str(x)[0])

preprocessed_sales = concat_sales_df.drop(['release_date'], axis=1)
preprocessed_sales = preprocessed_sales.merge(meta, on='item_number_color')

preprocessed_sales = preprocessed_sales.merge(product_codes, on='category')
preprocessed_sales['category'] = preprocessed_sales['meaning']
preprocessed_sales = preprocessed_sales.drop(['meaning'], axis=1)

preprocessed_sales['combination'] = preprocessed_sales['brand'] +"_"+ preprocessed_sales['fabric']  #조합 구성
preprocessed_sales.drop(['brand',	'main_color',	'fabric',	'category'], axis=1, inplace=True) 

#train, valid, test 분리
train_name_df = pd.DataFrame({"item_number_color": train})
train_df = preprocessed_sales.merge(train_name_df, on="item_number_color")

valid_name_df = pd.DataFrame({"item_number_color":valid})
valid_df = preprocessed_sales.merge(valid_name_df, on="item_number_color")

test_name_df = pd.DataFrame({"item_number_color":test})
test_df = preprocessed_sales.merge(test_name_df, on="item_number_color")


# train에서 'brand', 'fabric' 조합에 대한 주차별 median 값 계산
grouped = train_df.groupby(['combination'])[weeks].median().reset_index()

## test에 train, valid 합친 데이터의 미디언을 넣기 위하여 새로 만든 그룹
grouped2 = pd.concat([train_df, valid_df],axis=0).groupby(['combination'])[weeks].median().reset_index()

# valid에서 'brand', 'fabric' 조합에 대한 주차별 median 값 계산
grouped3 = valid_df.groupby(['combination'])[weeks].median().reset_index()

# train에서 각 item_number_color에 대해 해당 조합의 median 값과 조합 정보 할당
train_df = train_df[['item_number_color', 'combination']].merge(grouped, on=['combination'], how='left', suffixes=('', '_median'))

# train 결과 확인 및 저장
print(train_df.shape)
train_df.to_csv("/home/jihwan/fashion/data/median_of_each_combination/brand&fabric/train_data/train_median_of_each_combination_(brand&fabric).csv", index=False)


###### valid, test에는 각 아이템이 속한 메타데이터 조합에 대해 train에서의 메타데이터 조합과 일치하는 경우 train에서의 그 조합의 주차별 값을 할당함. #######

# valid 데이터에 대해 train의 메타데이터 조합과 일치하는 경우 train의 주차별 median값을 할당
valid_df = valid_df[['item_number_color', 'combination']].merge(grouped3, on=['combination'], how='left', suffixes=('', '_median'))


# ### 결측치를 제외한 나머지 행들의 각 주차별 평균을 결측치에 넣음 - 처음에 주석처리해서 테이블 만들고 결측값 확인 후 주석 제거해서 결측값 인덱스 넣어서 다시 코드 실행해야 함
# # missing_value의 인덱스를 제외한 나머지 행들의 각 주차별 평균 계산
# week_means = valid_df[~valid_df.index.isin([16, 709, 729, 974, 975, 1852, 1853])][weeks].mean().round(1)

# # 결측치가 있는 행들의 week1~week12 값을 평균값으로 대체
# for index in [16, 709, 729, 974, 975, 1852, 1853]:
#     valid_df.loc[index, weeks] = week_means.values

# valid 결과 확인 및 저장
print(valid_df.shape)
valid_df.to_csv("/home/jihwan/fashion/data/median_of_each_combination/brand&fabric/valid_data_towhich_train_value_was_added/valid_median_of_each_combination_(brand&fabric).csv", index=False)

# test 데이터에 대해 train의 메타데이터 조합과 일치하는 경우 주차별 값을 할당
# test_df = test_df[['item_number_color', 'combination']].merge(grouped, on=['combination'], how='left',  suffixes=('', '_median'))
test_df = test_df[['item_number_color', 'combination']].merge(grouped, on=['combination'], how='left',  suffixes=('', '_median'))

# ### 결측치를 제외한 나머지 행들의 각 주차별 평균을 결측치에 넣음- 처음에 주석처리해서 테이블 만들고 결측값 확인 후 주석 제거해서 결측값 인덱스 넣어서 다시 코드 실행해야 함
# # missing_value의 인덱스를 제외한 나머지 행들의 각 주차별 평균 계산
# week_means = test_df[~test_df.index.isin([39, 40, 156, 160, 161, 640, 1583, 2093, 2134, 2135, 2450])][weeks].mean().round(1)

# # 결측치가 있는 행들의 week1~week12 값을 평균값으로 대체
# for index in [39, 40, 156, 160, 161, 640, 1583, 2093, 2134, 2135, 2450]:
#     test_df.loc[index, weeks] = week_means.values

# test 결과 확인 및 저장
print(test_df.shape)
test_df.to_csv("/home/jihwan/fashion/data/median_of_each_combination/brand&fabric/test_data_towhich_train_value_was_added/test_median_of_each_combination_(brand&fabric).csv", index=False)


(14757, 14)
(1919, 14)
(2822, 14)


In [38]:
#Valid 처리 결과 - Train의 값이 들어가야 함
valid_after = pd.read_csv("/home/jihwan/fashion/data/median_of_each_combination/brand&fabric/valid_data_towhich_train_value_was_added/valid_median_of_each_combination_(brand&fabric).csv")
print(valid_after.isnull().sum())

# null 값이 포함된 행의 인덱스 찾기
null_indices = valid_after[valid_after.isnull().any(axis=1)].index
print("null indices:",null_indices)

valid_after

item_number_color    0
combination          0
week1                0
week2                0
week3                0
week4                0
week5                0
week6                0
week7                0
week8                0
week9                0
week10               0
week11               0
week12               0
dtype: int64
null indices: Index([], dtype='int64')


,item_number_color,combination,week1,week2,week3,week4,week5,week6,week7,week8,week9,week10,week11,week12
0,JVRF726AKH,J_특종,6.0,10.0,8.0,7.0,9.0,10.0,7.0,6.0,7.0,6.0,4.0,5.0
1,JWBL221ABL,J_WOVEN,5.0,7.0,8.0,7.0,7.0,7.0,7.0,7.0,6.0,6.0,5.0,5.0
2,JWBL226ANA,J_WOVEN,5.0,7.0,8.0,7.0,7.0,7.0,7.0,7.0,6.0,6.0,5.0,5.0
3,JWBL322ANA,J_WOVEN,5.0,7.0,8.0,7.0,7.0,7.0,7.0,7.0,6.0,6.0,5.0,5.0
4,JWBL322CNA,J_WOVEN,5.0,7.0,8.0,7.0,7.0,7.0,7.0,7.0,6.0,6.0,5.0,5.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1914,MXWS0150MT,M_WOVEN,8.0,16.0,21.0,26.0,32.0,38.0,41.0,43.0,42.0,40.0,36.0,32.0
1915,MXWS0150SB,M_WOVEN,8.0,16.0,21.0,26.0,32.0,38.0,41.0,43.0,42.0,40.0,36.0,32.0
1916,MXWS0150WH,M_WOVEN,8.0,16.0,21.0,26.0,32.0,38.0,41.0,43.0,42.0,40.0,36.0,32.0
1917,MXWS0150WY,M_WOVEN,8.0,16.0,21.0,26.0,32.0,38.0,41.0,43.0,42.0,40.0,36.0,32.0


In [74]:
#Test 처리 결과 - Train의 값이 들어가야 함
test_after = pd.read_csv("/home/jihwan/fashion/data/median_of_each_combination/brand&fabric/test_data_towhich_train_value_was_added/test_median_of_each_combination_(brand&fabric).csv")
print(test_after.isnull().sum())


# null 값이 포함된 행의 인덱스 찾기
null_indices = test_after[test_after.isnull().any(axis=1)].index
print("null indices:",null_indices)

test_after

item_number_color    0
combination          0
week1                0
week2                0
week3                0
week4                0
week5                0
week6                0
week7                0
week8                0
week9                0
week10               0
week11               0
week12               0
dtype: int64
null indices: Index([], dtype='int64')


,item_number_color,combination,week1,week2,week3,week4,week5,week6,week7,week8,week9,week10,week11,week12
0,JXBL121ALY,J_WOVEN,5.0,7.0,8.0,7.0,7.0,7.0,7.0,7.0,6.0,6.0,5.0,5.0
1,JXBL121BLC,J_WOVEN,5.0,7.0,8.0,7.0,7.0,7.0,7.0,7.0,6.0,6.0,5.0,5.0
2,JXBL121BOW,J_WOVEN,5.0,7.0,8.0,7.0,7.0,7.0,7.0,7.0,6.0,6.0,5.0,5.0
3,JXBL121CYE,J_WOVEN,5.0,7.0,8.0,7.0,7.0,7.0,7.0,7.0,6.0,6.0,5.0,5.0
4,JXBL321AIN,J_DENIM,6.0,10.0,9.0,10.0,9.0,9.0,8.5,9.0,8.0,7.0,6.0,6.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2817,MYWS3221OR,M_WOVEN,8.0,16.0,21.0,26.0,32.0,38.0,41.0,43.0,42.0,40.0,36.0,32.0
2818,MYWS3230BR,M_WOVEN,8.0,16.0,21.0,26.0,32.0,38.0,41.0,43.0,42.0,40.0,36.0,32.0
2819,MYWS3230CH,M_WOVEN,8.0,16.0,21.0,26.0,32.0,38.0,41.0,43.0,42.0,40.0,36.0,32.0
2820,MYWS3231BL,M_DENIM,7.0,13.0,15.0,17.0,18.0,19.0,21.0,25.0,23.0,24.0,23.0,20.0


## brand & main_color & category 의 결측값을 처리할 수 있는 방안
1. brand & category에는 결측값이 없으므로 이 brand & category의 값을 brand & main_color & category의 결측값을 대체함
2. 단순하게 현재 median값으로 채워진 데이터프레임에서 각 주차별 평균으로 결측값을 대체함 -> 이것으로 채택하였음
3. 결측값을 제거함

## 훈련데이터 원본 비교(아래)

In [21]:
import pandas as pd
import numpy as np
import pickle

# 데이터 로드
with open("/home/jihwan/fashion/data/train_val_test_split/train_list_240916.pkl", 'rb') as f:
    train = pickle.load(f)

meta = pd.read_csv("/home/jihwan/fashion/data/total_meta_data_str_value_240916.csv")
meta['item_number_color'] = meta['Unnamed: 0']
meta.drop(['color_full_spell', 'Unnamed: 0'], axis=1, inplace=True)

concat_sales_df = pd.read_csv("/home/jihwan/fashion/data/total_r_s.csv")
product_codes = pd.read_csv("/home/jihwan/fashion/data/product_codes.csv", encoding='cp949')
weeks = ['week'+str(i+1) for i in range(0, 12)]

# 데이터 전처리
concat_sales_df_a = concat_sales_df[['item_number_color', 'release_date']]
concat_sales_df_b = concat_sales_df.drop(['item_number_color', 'release_date'], axis=1)
concat_sales_df_b.columns = weeks
concat_sales_df = pd.concat([concat_sales_df_a, concat_sales_df_b], axis=1)
concat_sales_df['brand'] = concat_sales_df['item_number_color'].apply(lambda x: str(x)[0])

preprocessed_sales = concat_sales_df.drop(['release_date'], axis=1)
preprocessed_sales = preprocessed_sales.merge(meta, on='item_number_color')

preprocessed_sales = preprocessed_sales.merge(product_codes, on='category')
preprocessed_sales['category'] = preprocessed_sales['meaning']
preprocessed_sales = preprocessed_sales.drop(['meaning'], axis=1)

train_name_df = pd.DataFrame({"item_number_color": train})
train_df = preprocessed_sales.merge(train_name_df, on="item_number_color")
train_df

,item_number_color,week1,week2,week3,week4,week5,week6,week7,week8,week9,week10,week11,week12,brand,main_color,fabric,category
0,JOBL16BAPK,1.0,0.0,0.0,0.0,2.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,J,PINK,WOVEN,BLOUSE
1,JOBL26FADN,11.0,8.0,3.0,7.0,3.0,2.0,2.0,1.0,2.0,2.0,4.0,0.0,J,BLUE,WOVEN,BLOUSE
2,JOBL26FAWH,9.0,7.0,3.0,10.0,3.0,7.0,4.0,3.0,1.0,3.0,2.0,0.0,J,WHITE,WOVEN,BLOUSE
3,JOBL361AOR,1.0,0.0,0.0,1.0,0.0,0.0,0.0,1.0,0.0,0.0,1.0,0.0,J,ORANGE,WOVEN,BLOUSE
4,JOBL361AWH,1.0,0.0,0.0,1.0,0.0,1.0,0.0,0.0,1.0,0.0,0.0,2.0,J,WHITE,WOVEN,BLOUSE
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
14752,MWWS0101BK,27.0,70.0,69.0,51.0,42.0,45.0,52.0,62.0,64.0,65.0,63.0,91.0,M,BLACK,WOVEN,W/SHIRTS
14753,MWWS0101DL,20.0,35.0,48.0,27.0,26.0,24.0,45.0,50.0,43.0,49.0,56.0,68.0,M,BLUE,WOVEN,W/SHIRTS
14754,MWWS0101LG,27.0,90.0,63.0,67.0,63.0,55.0,91.0,142.0,98.0,115.0,147.0,179.0,M,GREY,WOVEN,W/SHIRTS
14755,MWWS0102TB,10.0,20.0,29.0,30.0,15.0,13.0,27.0,29.0,33.0,30.0,44.0,48.0,M,BLUE,WOVEN,W/SHIRTS
